# TD3 baseline on Four Rooms

In [1]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

# Resolve repository src path robustly when running notebook in-place.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from environments.fourrooms import FourRoomsGridWorld, FourRoomsGoalWrapper
from utils import TrajectoryReplayBuffer, collect_episode, evaluate_policy
from visualisations import plot_policy_rollouts, plot_q_diagnostics
from networks import TD3_Critic
from agents import TD3_Actor
from benchmarks.factory import make_env, load_dataset_into_buffer

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)


Using device: mps


In [2]:
def make_pointmaze_env(
    task="point_mass_maze_reach_top_left",
    seed=0,
    render_mode=None,
):
    return make_env(
        benchmark="exorl",
        task=task,
        seed=seed,
        add_goal_wrapper=False,   # or True if you want GoalInfoWrapper
        render_mode=render_mode,
    )

In [3]:


def td3_train_offline(
    task="point_mass_maze_reach_top_left",
    dataset_domain="point_mass_maze",
    dataset_algorithm="random",
    seed=0,
    total_gradient_steps=300000,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    policy_noise=0.2,
    noise_clip=0.5,
    policy_delay=4,
    lr=3e-4,
    replay_capacity=2_000_000,
    eval_every=3000,
    eval_episodes=8,
):
    env = make_pointmaze_env(
        task=task,
        seed=seed,
        render_mode=None,
    )
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]

    actor = TD3_Actor(obs_dim, act_dim).to(DEVICE)
    actor_tgt = TD3_Actor(obs_dim, act_dim).to(DEVICE)
    actor_tgt.load_state_dict(actor.state_dict())

    q1 = TD3_Critic(obs_dim, act_dim).to(DEVICE)
    q2 = TD3_Critic(obs_dim, act_dim).to(DEVICE)
    q1_tgt = TD3_Critic(obs_dim, act_dim).to(DEVICE)
    q2_tgt = TD3_Critic(obs_dim, act_dim).to(DEVICE)
    q1_tgt.load_state_dict(q1.state_dict())
    q2_tgt.load_state_dict(q2.state_dict())

    actor_opt = optim.Adam(actor.parameters(), lr=lr)
    critic_opt = optim.Adam(list(q1.parameters()) + list(q2.parameters()), lr=lr)

    replay = TrajectoryReplayBuffer(
        capacity=replay_capacity,
        obs_dim=obs_dim,
        action_dim=act_dim,
        device=DEVICE,
    )

    num_loaded = load_dataset_into_buffer(
        benchmark="exorl",
        replay_buffer=replay,
        task=dataset_domain,
        algorithm=dataset_algorithm,
        auto_download=True,
        max_episodes=None,
    )
    print("Loaded episodes:", num_loaded)

    eval_returns = []

    for train_it in range(1, total_gradient_steps + 1):
        batch = replay.sample(batch_size)
        obs = batch.obs.float()
        act = batch.actions.float()
        rew = batch.rewards.float()
        next_obs = batch.next_obs.float()
        done = torch.clamp(batch.terminated + batch.truncated, 0, 1).float()

        with torch.no_grad():
            noise = torch.randn_like(act) * policy_noise
            noise = torch.clamp(noise, -noise_clip, noise_clip)
            next_a = torch.clamp(actor_tgt(next_obs) + noise, -1.0, 1.0)
            target_q = torch.min(q1_tgt(next_obs, next_a), q2_tgt(next_obs, next_a))
            y = rew + gamma * (1.0 - done) * target_q

        critic_loss = F.mse_loss(q1(obs, act), y) + F.mse_loss(q2(obs, act), y)
        critic_opt.zero_grad()
        critic_loss.backward()
        critic_opt.step()

        if train_it % policy_delay == 0:
            actor_loss = -q1(obs, actor(obs)).mean()
            actor_opt.zero_grad()
            actor_loss.backward()
            actor_opt.step()

            with torch.no_grad():
                for p, pt in zip(actor.parameters(), actor_tgt.parameters()):
                    pt.data.mul_(1 - tau).add_(tau * p.data)
                for p, pt in zip(q1.parameters(), q1_tgt.parameters()):
                    pt.data.mul_(1 - tau).add_(tau * p.data)
                for p, pt in zip(q2.parameters(), q2_tgt.parameters()):
                    pt.data.mul_(1 - tau).add_(tau * p.data)

        if train_it % eval_every == 0:
            eval_env = make_pointmaze_env(
                task=task,
                seed=seed,
                render_mode=None,
            )
            mean_ret, mean_len = evaluate_policy(
                eval_env,
                lambda o: actor(
                    torch.tensor(o, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                ).squeeze(0).detach().cpu().numpy().astype(np.float32),
                episodes=eval_episodes,
            )
            eval_returns.append((train_it, mean_ret))
            print(
                f"[TD3-offline] grad_step={train_it:7d} | "
                f"eval_return={mean_ret:.3f} | eval_len={mean_len:.1f}"
            )
            eval_env.close()

    env.close()
    return actor, q1, q2, eval_returns


td3_actor, td3_q1, td3_q2, td3_eval = td3_train_offline(
    task="point_mass_maze_reach_top_left",
    dataset_domain="point_mass_maze",
    dataset_algorithm="random",   # try "aps", "rnd", "proto" later
    seed=0,
    total_gradient_steps=300000,
)

if td3_eval:
    xs, ys = zip(*td3_eval)
    plt.figure(figsize=(7, 4))
    plt.plot(xs, ys)
    plt.xlabel('Environment steps')
    plt.ylabel('Mean episodic return')
    plt.title('TD3 baseline on PointMass Maze')
    plt.grid(alpha=0.25)
    plt.show()


NotImplementedError: This repo only provides dm_control-backed ExORL environments for suite tasks. Supported tasks: cartpole_balance, cartpole_balance_sparse, cartpole_swingup, cartpole_swingup_sparse, cheetah_run, quadruped_run, quadruped_walk, walker_run, walker_stand, walker_walk. Requested: point_mass_maze_reach_top_left.

## Visualisations

In [ ]:
from benchmarks.visualise import collect_episode_frames, save_gif
from IPython.display import Image, display

eval_env = make_pointmaze_env(
    task="point_mass_maze_reach_top_left",
    seed=0,
    render_mode="rgb_array",
)

def td3_policy(env, obs):
    del env
    obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    with torch.no_grad():
        return td3_actor(obs_t).squeeze(0).cpu().numpy().astype(np.float32)

rollout = collect_episode_frames(eval_env, policy_fn=td3_policy, max_steps=500)
save_gif(rollout["frames"], "td3_pointmaze.gif", fps=30)

print("Steps:", rollout["num_steps"])
print("Return:", rollout["total_reward"])

display(Image(filename="td3_pointmaze.gif"))
eval_env.close()